In [2]:
import os, random
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

In [ ]:
def set_seed(int=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def build_dataset(tokenizer, train_path, valid_path, max_length=128):
    def tokenize_fn(example):
        return tokenizer(example["text"], truncation=True, padding="max_length", max_length=max_length)

    # ds = load_datset("imdb")
    ds = load_dataset("csv", data_files={"train": train_path, "valid": valid_path})

    train_ds = ds["train"].map(tokenize_fn, batched=True)
    val_ds = ds["val"].map(tokenize_fn, batched=True)

    train_tok = train_ds.rename_column("label", "labels")
    val_tok = val_ds.rename_column("label", "labels")

    train_tok.set_format(type="torch", columns_name=["input_ids", "attention_mask", "labels"])
    val_tok.set_format(type="torch", columns_name=["input_ids", "attention_mask", "labels"])

    return train_tok, val_tok

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy.compute(predictions=preds, references=labels)
    f1 = f1.compute(predictions=preds, references=labels, average=binary)
    return acc, f1

def train_fn(model_name="distilbert-base-uncased", output_dir="./result", num_labels=2):
    set_seed(42)
    device = torch.device("cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name) 
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)
    
    train, val = build_dataset(tokenizer, train_path, valid_path, max_length=128)

    args = TrainingArguments(
        output_dir=output_dir, eval_strategy="epoch", num_train_epochs=1,
        per_device_train_batch_size=4, per_device_eval_batch_size=4, learning_rate=2e-5,
        save_strategy="no", logging_strategy="steps", logging_steps=10)

    trainer = Trainer(
        model=model, args=args, train_dataset=train, eval_dataset=val, processing_class=tokenizer, compute_metrics=compute_metrics)

    trainer.train()

    metrics = trainer.evaluate()
    
    return metrics
    